# Representative covers and components

Choose diverse examples with representative covers, and inspect related groups with component IDs. Covers select representatives; components connect items through the full similarity graph.

In [ ]:
import pandas as pd

from plinder.core import PlinderRelease, query_table
from plinder.core.scores import query_ligand_similarity

release = PlinderRelease()
ligand_id = "4agi__1__1.R"
cover = "pocket_qcov__70__ligand__directed_set_cover"
component = "pocket_qcov__70__ligand__reciprocal_component"

## Find a ligand's representative group

Cover columns join automatically onto ligand annotations. `__is_centroid` identifies published representatives.

In [ ]:
query_table(
    columns=["system_id", "ligand_id", cover, f"{cover}__is_centroid", component],
    filters=[("ligand_id", "==", ligand_id)],
    release=release,
).T

## Inspect the actual assignment threshold

Directional covers can use the fallback threshold for members that lack a match at the requested threshold. The detailed cover table records the representative, similarity, and threshold used for each assignment.

In [ ]:
details = pd.read_parquet(
    release.fetch("ligand_sampling")
    / "directed_set_cover"
    / "metric=pocket_qcov"
    / "threshold=70.parquet",
    columns=[
        "ligand_id",
        "centroid_ligand_id",
        "similarity_to_centroid",
        "assignment_threshold",
        "representative_selection_threshold",
    ],
    filters=[("ligand_id", "==", ligand_id)],
)
details

In [ ]:
query_table(
    columns=["ligand_id", "system_id", "ligand_unique_ccd_code", "entry_resolution"],
    filters=[("ligand_id", "in", details["centroid_ligand_id"].unique().tolist())],
    release=release,
)

## Whole-interface and half-interface groups

Whole-interface columns use `interface_qcov`; each side has its own `interface_side_qcov` columns. Reciprocal components use edges meeting the threshold in both directions. Members can connect through other members.

In [ ]:
query_table(
    "interface_annotations",
    columns=[
        "system_id",
        "interface_qcov__70__directed_set_cover",
        "interface_qcov__70__reciprocal_component",
        "interface_side_qcov__70__chain_1_directed_set_cover",
        "interface_side_qcov__70__chain_2_directed_set_cover",
        "interface_side_qcov__70__chain_1_reciprocal_component",
        "interface_side_qcov__70__chain_2_reciprocal_component",
    ],
    filters=[("entry_pdb_id", "==", "7cm8")],
    release=release,
).head()

## Protein sequence and structure clusters

Protein clusters identify their representatives by PDB entry and label asym ID. The status column records how each chain was assigned.

In [ ]:
for table in ("protein_sequence_clusters", "protein_structure_clusters"):
    display(
        query_table(
            table,
            filters=[("entry_pdb_id", "in", ["4agi", "7eek"])],
            release=release,
        )
    )

## Screen a proposed holdout

Here, `7eek` is a small illustrative holdout. We remove candidate training systems containing a ligand in any of its pocket components. Use the full candidate set and the metrics relevant to your benchmark for a real split.

In [ ]:
candidates = query_table(
    columns=["entry_pdb_id", "system_id", "ligand_id", component],
    filters=[
        ("entry_pdb_id", "in", ["4agi", "4agc", "7eek"]),
        ("ligand_is_proper", "==", True),
    ],
    release=release,
).dropna(subset=[component])
heldout = candidates.loc[candidates["entry_pdb_id"].eq("7eek")]
heldout_components = set(heldout[component])
blocked_systems = set(
    candidates.loc[candidates[component].isin(heldout_components), "system_id"]
)
training = candidates.loc[
    ~candidates["entry_pdb_id"].eq("7eek")
    & ~candidates["system_id"].isin(blocked_systems)
]
pd.Series(
    {
        "heldout_systems": heldout["system_id"].nunique(),
        "remaining_training_systems": training["system_id"].nunique(),
    }
)

## Check directional similarities across the sets

Inspect both query directions as a separate screen. Reciprocal components capture one metric's reciprocal graph; a benchmark can also need chemical, sequence, interface, or one-way similarity criteria.

In [ ]:
cross_scores = pd.concat(
    [
        query_ligand_similarity(
            columns=["query_ligand_id", "target_ligand_id", "pocket_qcov", "pli_qcov"],
            filters=[
                ("query_ligand_id", "in", queries["ligand_id"].tolist()),
                ("target_ligand_id", "in", targets["ligand_id"].tolist()),
            ],
            release=release,
        )
        for queries, targets in [(training, heldout), (heldout, training)]
    ],
    ignore_index=True,
)
cross_scores.sort_values("pocket_qcov", ascending=False).head()

For the original dataset-selection and similarity exercises, see the [Moving Beyond Memorization workshop](https://github.com/plinder-org/moving_beyond_memorisation). This guide uses the current release tables and API.